# Dam-break visualization with the amplitude-based collisionless QLBM

In [ ]:
from qiskit_aer import AerSimulator

from qlbm.components import (
    CQLBM,
    ABDiscreteUniformInitialConditions,
    ABGridMeasurement,
    EmptyPrimitive,
)
from qlbm.infra import QiskitRunner, SimulationConfig
from qlbm.lattice import ABLattice
from qlbm.tools.utils import create_directory_and_parents

In [ ]:
lattice = ABLattice(
    {
        "lattice": {"dim": {"x": 16, "y": 16}, "velocities": "d2q9"},
        "geometry": [
            {"shape": "cuboid", "x": [9, 10], "y": [0, 5], "boundary": "bounceback"},
            {"shape": "cuboid", "x": [9, 10], "y": [10, 15], "boundary": "bounceback"},
        ],
    }
)

output_dir = "qlbm-output/ab-d2q9-dam-break"
create_directory_and_parents(output_dir)

In [ ]:
initial_conditions = ABDiscreteUniformInitialConditions(
    lattice,
    [1, 5, 8],  # East, northeast, and southeast
    ([0, 1], list(range(4))),
).circuit
initial_conditions.x(lattice.grid_index(0)[2])  # Shift reservoir to x=4–7

cfg = SimulationConfig(
    initial_conditions=initial_conditions,
    algorithm=CQLBM(lattice),
    postprocessing=EmptyPrimitive(lattice),
    measurement=ABGridMeasurement(lattice),
    target_platform="QISKIT",
    compiler_platform="QISKIT",
    optimization_level=0,
    statevector_sampling=True,
    execution_backend=AerSimulator(method="statevector", max_parallel_threads=2, seed_simulator=42),
    sampling_backend=AerSimulator(method="statevector", max_parallel_threads=2, seed_simulator=42),
)

In [ ]:
cfg.prepare_for_simulation()

In [ ]:
# Number of shots to simulate for each timestep when running the circuit
NUM_SHOTS = 2**12

# Number of timesteps to simulate
NUM_STEPS = 8

In [ ]:
runner = QiskitRunner(
    cfg,
    lattice,
)


# Simulate the circuits using both snapshots
runner.run(
    NUM_STEPS,  # Number of time steps
    NUM_SHOTS,  # Number of shots per time step
    output_dir,
    statevector_snapshots=True,
)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Image, display
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Rectangle
from vtk import vtkXMLImageDataReader
from vtk.util.numpy_support import vtk_to_numpy

frames = []
for step in range(NUM_STEPS + 1):
    reader = vtkXMLImageDataReader()
    reader.SetFileName(str(Path(output_dir) / "paraview" / f"step_{step:03d}.vti"))
    reader.Update()
    nx, ny, _ = reader.GetOutput().GetDimensions()
    counts = vtk_to_numpy(reader.GetOutput().GetPointData().GetScalars())
    frames.append(counts.reshape(ny, nx) / NUM_SHOTS)

fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
image = ax.imshow(
    frames[0],
    origin="lower",
    interpolation="nearest",
    cmap="Blues",
    vmin=0,
    vmax=max(frame.max() for frame in frames),
)
for obstacle in json.loads(lattice.to_json())["geometry"]:
    x0, x1 = obstacle["x"]
    y0, y1 = obstacle["y"]
    ax.add_patch(Rectangle(
        (x0 - 0.5, y0 - 0.5), x1 - x0 + 1, y1 - y0 + 1,
        facecolor="lightgray", edgecolor="black", label="Dam wall" if y0 == 0 else None,
    ))
ax.set(xlabel="x", ylabel="y", xticks=range(0, nx, 3), yticks=range(0, ny, 3))
ax.legend(loc="upper left")
ax.text(5.5, 14.4, "Reservoir", ha="center", fontsize=8)
ax.text(13, 14.4, "Downstream", ha="center", fontsize=8)
fig.colorbar(image, ax=ax, label="Measured position probability")
title = ax.set_title("Dam release · step 00")

def draw_step(step):
    image.set_data(frames[step])
    title.set_text(f"Dam release · step {step:02d}")

gif_path = Path(output_dir) / "flow.gif"
animation = FuncAnimation(fig, draw_step, frames=len(frames))
animation.save(gif_path, writer=PillowWriter(fps=2), dpi=120)
plt.close(fig)
display(Image(filename=str(gif_path)))